In [ ]:
import gzip
from multiprocessing import Pool
from pathlib import Path

import pandas as pd
from tqdm import tqdm

In [ ]:
class VisualizerReader:

    def __init__(self, f, msf_prep_time: int):
        self.f = f
        self.msf_prep_time = msf_prep_time

    def read_header(self):
        f = self.f
        self.width, self.height, self.layer_count = map(int, f.readline().split())
        self.data_qubit_count, self.msf_count = map(int, f.readline().split())
        self.data_qubits = []
        for _ in range(self.data_qubit_count):
            self.data_qubits.append(tuple(map(int, f.readline().split())))
        self.ms_factories = []
        for _ in range(self.msf_count):
            self.ms_factories.append(tuple(map(int, f.readline().split())))
        self.position2qubit_index = {t: i for i, t in enumerate(self.data_qubits)} | {
            t: i for i, t in enumerate(self.ms_factories, start=self.data_qubit_count)
        }
        self.path_count, self.max_timing = map(int, f.readline().split())

    def read_path(self):
        f = self.f
        target_count = int(f.readline())
        targets = list(map(int, f.readline().split()))
        path_length = int(f.readline())
        path = [tuple(map(int, f.readline().split())) for _ in range(path_length)]
        return (targets, path)

    def compute_metrics(self):
        self.read_header()

        total_code_beat = self.max_timing + 1
        circuit_volume = total_code_beat * self.data_qubit_count
        for _ in range(self.path_count):
            targets, path = self.read_path()
            if len(path) >= 2:
                circuit_volume += len(path) - 2
            elif len(path) == 1:  # if No_MSF and MSF operation is involved
                circuit_volume += self.msf_prep_time + 1
            for target_id in targets:
                if target_id >= self.data_qubit_count:
                    circuit_volume += self.msf_prep_time

        assert not self.f.readline()

        metrics = dict()
        metrics["total_code_beat"] = total_code_beat
        metrics["circuit_volume"] = circuit_volume

        return metrics

In [ ]:
row_arguments = []

result_path = Path("../../out/result")
circuit_paths = result_path.glob("result_trotter_10_Heisenberg*")

for circuit_path in sorted(circuit_paths):
    circuit_name = circuit_path.name
    param_paths = list(circuit_path.glob("*_2"))
    # param_paths = sum((list(circuit_path.glob(f"*_{t}")) for t in range(0, 11, 2)), [])
    for param_path in sorted(param_paths):
        param_name = param_path.name
        routing_files = param_path.glob("*")
        for routing_file in sorted(routing_files):
            # Note that the following removes ".txt" or ".txt.gz"
            routing_name = routing_file.name.partition(".")[0]
            row_arguments.append((circuit_name, param_name, routing_name, routing_file))

In [ ]:
row_arguments

[('result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det',
  'inner_1_SA_0.001_2',
  'Double_No_Kink',
  PosixPath('../../out/result/result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det/inner_1_SA_0.001_2/Double_No_Kink.txt.gz')),
 ('result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det',
  'inner_1_SA_0.001_2',
  'Double_No_MSF',
  PosixPath('../../out/result/result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det/inner_1_SA_0.001_2/Double_No_MSF.txt.gz')),
 ('result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det',
  'inner_1_SA_0.001_2',
  'Double_No_Top',
  PosixPath('../../out/result/result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det/inner_1_SA_0.001_2/Double_No_Top.txt.gz')),
 ('result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det',
  'inner_1_SA_0.001_2',
  'Double_Yes_Kink',
  PosixPath('../../out/result/result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det/inner_1_SA_0.001_2/Double_Yes_Kink.txt.gz')),
 ('result_trotter_10_Heisenberg2D_cylinder_0.5_0.5_det',
  'inner_1_SA_0.001_2',
  'Proj_N

In [ ]:
def compute_row_dict(row_argument):
    circuit_name, param_name, routing_name, routing_file = row_argument
    row = dict()
    row["circuit"] = circuit_name
    (
        row["factory"],
        row["layer_count"],
        row["allocator"],
        row["msf_coeff"],
        row["msf_prep_time"],
    ) = param_name.split("_")
    row["routing_algo"], _, row["routing_option"] = routing_name.partition("_")
    if routing_file.with_suffix(".gz"):
        with gzip.open(routing_file, "rb") as f:
            row |= VisualizerReader(f, int(row["msf_prep_time"])).compute_metrics()
    else:
        with open(routing_file, "r") as f:
            row |= VisualizerReader(f, int(row["msf_prep_time"])).compute_metrics()

    return row


list_of_row_dict = []
with Pool(16) as p:
    for d in tqdm(
        p.imap_unordered(compute_row_dict, row_arguments), total=len(row_arguments)
    ):
        list_of_row_dict.append(d)

100%|██████████| 552/552 [00:21<00:00, 25.40it/s]


In [ ]:
df = pd.DataFrame(list_of_row_dict)
df.to_csv("../../out/table/Heisenberg2D_10_Trotter_.csv", index=False)
df

,circuit,factory,layer_count,allocator,msf_coeff,msf_prep_time,routing_algo,routing_option,total_code_beat,circuit_volume
0,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,inner,1,SA,0.001,2,Single,No_MSF,756,152100
1,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,inner,1,SA,0.001,2,Proj,No_MSF,390,115500
2,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,inner,1,SA,0.001,2,Double,No_MSF,749,151400
3,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,inner,1,SA,0.01,2,Double,No_Top,1861,237100
4,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,inner,1,SA,0.001,2,Double,No_Top,1861,237100
...,...,...,...,...,...,...,...,...,...,...
547,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,outer,2,random,0,2,Single,Yes_Kink,1836,516587
548,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,outer,2,naive,0,2,Proj,No_Kink,1936,1136201
549,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,outer,2,naive,0,2,Proj,Yes_Kink,1939,1116237
550,result_trotter_10_Heisenberg2D_cylinder_0.5_0....,outer,2,random,0,2,Proj,Yes_Kink,1935,1121133
